In [ ]:
!pip install -q fastapi uvicorn peft transformers accelerate pillow pyngrok nest_asyncio bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
LORA_PATH = '/content/drive/MyDrive/Colab Notebooks/VU_DL_Team_Project/outputs/v3_mild_oversample/final'
print(os.listdir(LORA_PATH))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']


In [ ]:
import shutil, os

os.makedirs('/content/addin/taskpane', exist_ok=True)

shutil.copy('/content/drive/MyDrive/TeamTask/addin/taskpane/taskpane.html', '/content/addin/taskpane/taskpane.html')
shutil.copy('/content/drive/MyDrive/TeamTask/addin/taskpane/taskpane.js', '/content/addin/taskpane/taskpane.js')
shutil.copy('/content/drive/MyDrive/TeamTask/addin/manifest.xml', '/content/addin/manifest.xml')

print(os.listdir('/content/addin/taskpane'))

In [ ]:
from huggingface_hub import login
login(token="YOUR TOKEN")

In [ ]:
with open('/content/app.py', 'w') as f:
    f.write("""import base64, io, re, os
from typing import Optional
import torch
from fastapi import FastAPI, HTTPException, Header
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from PIL import Image, ImageFilter
from pydantic import BaseModel
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
from transformers import BitsAndBytesConfig

LORA_PATH   = "/content/drive/MyDrive/Colab Notebooks/VU_DL_Team_Project/outputs/v3_mild_oversample/final"
MODEL_ID    = "google/paligemma2-3b-pt-448"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
PROMPT      = "detect sensitive\\n"
BLUR_RADIUS = 20
SECRET_KEY  = "demo-secret-2024"

print(f"Loading on {DEVICE}...")
processor = AutoProcessor.from_pretrained(LORA_PATH)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)
base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()
print("Model ready!")

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["POST", "GET"], allow_headers=["*"])

class BlurRequest(BaseModel):
    image_b64: str
    filename: Optional[str] = "attachment.png"

class BoundingBox(BaseModel):
    x1: int; y1: int; x2: int; y2: int
    label: str

class BlurResponse(BaseModel):
    blurred_image_b64: str
    boxes: list[BoundingBox]
    original_width: int
    original_height: int

def load_image(b64):
    if "," in b64:
        b64 = b64.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")

def run_inference(image):
    inputs = processor(text=PROMPT, images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    prompt_len = inputs["input_ids"].shape[1]
    return processor.decode(out[0][prompt_len:], skip_special_tokens=False)

def parse_boxes(text, w, h):
    boxes = []
    pattern = re.compile(r"<loc(\\d{4})><loc(\\d{4})><loc(\\d{4})><loc(\\d{4})>\\s*(\\w+)?")
    for m in pattern.finditer(text):
        ny1, nx1, ny2, nx2 = (int(m.group(i)) for i in range(1, 5))
        label = m.group(5) or "sensitive"
        boxes.append(BoundingBox(
            x1=int(nx1/1024*w), y1=int(ny1/1024*h),
            x2=int(nx2/1024*w), y2=int(ny2/1024*h),
            label=label,
        ))
    return boxes

def apply_blur(image, boxes):
    result = image.copy()
    for b in boxes:
        region = result.crop((b.x1, b.y1, b.x2, b.y2))
        result.paste(region.filter(ImageFilter.GaussianBlur(BLUR_RADIUS)), (b.x1, b.y1))
    return result

def to_b64(image):
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

@app.post("/process-image", response_model=BlurResponse)
async def process_image(req: BlurRequest, x_api_key: str = Header(...)):
    if x_api_key != SECRET_KEY:
        raise HTTPException(403, "Forbidden")
    try:
        image = load_image(req.image_b64)
    except Exception as e:
        raise HTTPException(400, f"Invalid image: {e}")
    w, h = image.size
    raw  = run_inference(image)
    print("Model output:", raw)
    boxes = parse_boxes(raw, w, h)
    return BlurResponse(
        blurred_image_b64=to_b64(apply_blur(image, boxes) if boxes else image),
        boxes=boxes,
        original_width=w,
        original_height=h,
    )

@app.get("/health")
async def health():
    return {"status": "ok", "device": DEVICE}

app.mount("/addin", StaticFiles(directory="/content/addin"), name="addin")
""")

app.py written!


In [ ]:
import nest_asyncio, uvicorn, threading, time
from pyngrok import ngrok

nest_asyncio.apply()

def run():
    uvicorn.run("app:app", host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=run, daemon=True).start()
time.sleep(10)  # wait for model to load + server to start

tunnel = ngrok.connect(8000)
print("✅ Backend URL:", tunnel.public_url)

Loading on cuda...
✅ Backend URL: https://grain-scheming-ellipse.ngrok-free.dev
